In [8]:
import numpy as np
import math

def a_alpha(alpha: float, tau: float) -> float:
    # 論文(11)の“ -1 ”まで含めた厳密版
    return 2.0 * ((tau/alpha**2)*(1.0-alpha) - 1.0)

def cubic_coeffs(alpha: float, tau: float, J: float, h: float):
    """
    式(13)を
      p3*m^3 + p2*m^2 + p1*m + p0 = 0
    に直した係数を返す
    """
    k = tau/alpha
    B = -4.0*k*J         # (N-1)b
    p3 = -B              # m^3の係数
    p2 = 2.0*k*h         # m^2の係数
    p1 = a_alpha(alpha, tau) + B
    p0 = -2.0*k*h
    return p3, p2, p1, p0

def discriminant(alpha: float, tau: float, J: float, h: float) -> float:
    # 三次式の判別式 Δ
    p3, p2, p1, p0 = cubic_coeffs(alpha, tau, J, h)
    Δ = (18*p3*p2*p1*p0
         - 4*(p2**3)*p0
         + (p2**2)*(p1**2)
         - 4*p3*(p1**3)
         - 27*(p3**2)*(p0**2))
    return Δ

def alpha_c_h(tau: float, J: float, h: float,
              alpha_lo: float = 0.7, alpha_hi: float = 0.99,
              nscan: int = 3000, iters: int = 80) -> float:
    """
    Δ(α)=0 を二分法で解く（符号変化をscanで見つけてから絞る）
    """
    alphas = np.linspace(alpha_lo, alpha_hi, nscan)
    vals = np.array([discriminant(a, tau, J, h) for a in alphas])

    # 符号変化区間を探す
    idx = np.where(vals[:-1]*vals[1:] < 0)[0]
    if len(idx) == 0:
        raise RuntimeError(f"No sign change found. Try widening range. "
                           f"(h={h}, tau={tau}, J={J})")

    i = idx[0]
    lo, hi = float(alphas[i]), float(alphas[i+1])
    flo = float(discriminant(lo, tau, J, h))

    # 二分法
    for _ in range(iters):
        mid = (lo + hi) / 2.0
        fmid = float(discriminant(mid, tau, J, h))
        if flo * fmid <= 0:
            hi = mid
        else:
            lo = mid
            flo = fmid
    return (lo + hi) / 2.0


In [13]:
J = 0.1
h = 1e-3

print("tau=100 :", alpha_c_h(tau=100,  J=J, h=h, alpha_lo=0.75, alpha_hi=0.9))
print("tau=1000:", alpha_c_h(tau=1000, J=J, h=h, alpha_lo=0.75, alpha_hi=0.9))


tau=100 : 0.8351182404794812
tau=1000: 0.8403894997081325
